In [ ]:
# Standard library
import warnings
from datetime import datetime, timezone
import platform

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset
from sklearn.datasets import fetch_california_housing

# Modeling
from sklearn.model_selection import (
    train_test_split, KFold, GroupKFold, cross_validate, GridSearchCV
)
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Evaluation
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.inspection import permutation_importance

# Persistence
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Do not globally suppress warnings: warnings can reveal compatibility
# or modeling issues that should be investigated.

## Catatan revisi

Notebook ini mempertahankan eksperimen awal, lalu menambahkan:
1. `DummyRegressor` sebagai baseline sederhana.
2. R² untuk melengkapi MAE dan RMSE.
3. Spatial cross-validation sebagai pemeriksaan tambahan karena data memiliki Latitude/Longitude.
4. Penjelasan target yang dibatasi (capped) dan batas interpretasi prediksi.
5. Metadata artifact dan validasi urutan fitur.
6. Kesimpulan yang mengambil angka langsung dari hasil eksekusi.

**Penting:** jalankan ulang seluruh notebook dari atas ke bawah (*Restart Kernel & Run All*). Output lama sengaja dibersihkan agar angka yang terlihat tidak disalahartikan sebagai hasil eksekusi versi revisi.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
quality = pd.DataFrame({
    "missing": df.isna().sum(),
    "duplicates": [df.duplicated().sum()] * df.shape[1],
    "dtype": df.dtypes,
})
quality

In [ ]:
df.hist(figsize=(15, 10), bins=50)
plt.suptitle("Distribution of Numeric Features", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), df.columns[:-1]):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, square=True)
plt.title("Correlation Matrix")
plt.show()

corr["MedHouseVal"].drop("MedHouseVal").sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x="MedInc", y="MedHouseVal", alpha=0.3, edgecolor=None)
plt.title("Median Income vs Median House Value")
plt.xlabel("Median Income (MedInc)")
plt.ylabel("Median House Value (MedHouseVal)")
plt.show()

### Catatan tentang target `MedHouseVal`

`MedHouseVal` adalah median house value pada tingkat *block group* dalam dataset California Housing, bukan harga transaksi satu rumah individual. Nilai target pada dataset juga memiliki batas atas sekitar 5.0 (dalam satuan US$100.000). Karena itu, prediksi sebaiknya dipahami sebagai estimasi nilai median wilayah berdasarkan fitur agregat, bukan appraisal rumah individual.

In [ ]:
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """Fit model and return train/test MAE, RMSE, and R²."""
    model.fit(X_tr, y_tr)
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)

    return {
        "Model": name,
        "MAE Train": mean_absolute_error(y_tr, y_tr_pred),
        "MAE Test": mean_absolute_error(y_te, y_te_pred),
        "RMSE Train": np.sqrt(mean_squared_error(y_tr, y_tr_pred)),
        "RMSE Test": np.sqrt(mean_squared_error(y_te, y_te_pred)),
        "R2 Train": r2_score(y_tr, y_tr_pred),
        "R2 Test": r2_score(y_te, y_te_pred),
        "_model": model,
    }

### Baseline: DummyRegressor

Baseline ini memprediksi nilai median target pada data training untuk setiap baris. Model yang lebih kompleks seharusnya mengungguli baseline ini pada data test; jika tidak, kompleksitas model belum terbukti memberi manfaat.

In [ ]:
dummy_result = evaluate_model(
    "DummyRegressor (median)",
    DummyRegressor(strategy="median"),
    X_train, y_train, X_test, y_test,
)
print({k: v for k, v in dummy_result.items() if k != "_model"})

In [ ]:
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression()),
])
lr_result = evaluate_model("Linear Regression", lr_pipeline, X_train, y_train, X_test, y_test)
print({k: v for k, v in lr_result.items() if k != "_model"})

In [ ]:
dt_result = evaluate_model(
    "Decision Tree (default)",
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    X_train, y_train, X_test, y_test,
)

dt_limited_result = evaluate_model(
    "Decision Tree (max_depth=10)",
    DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE),
    X_train, y_train, X_test, y_test,
)

In [ ]:
rf_result = evaluate_model(
    "Random Forest (baseline)",
    RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    X_train, y_train, X_test, y_test,
)

In [ ]:
results = pd.DataFrame([
    {k: v for k, v in r.items() if k != "_model"}
    for r in [dummy_result, lr_result, dt_result, dt_limited_result, rf_result]
])
results.sort_values("MAE Test").reset_index(drop=True)

### Hyperparameter tuning

`GridSearchCV` hanya menerima `X_train` dan `y_train`. Test set tetap disimpan untuk evaluasi akhir. Karena beberapa kandidat dibandingkan pada cross-validation, skor CV terbaik dari GridSearch adalah skor seleksi, bukan estimasi final yang sepenuhnya bebas bias seleksi.

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV MAE:", -grid_search.best_score_)

In [ ]:
best_model = grid_search.best_estimator_
tuned_result = evaluate_model("Random Forest (tuned)", best_model,
                              X_train, y_train, X_test, y_test)
print({k: v for k, v in tuned_result.items() if k != "_model"})

In [ ]:
fi_impurity = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_model.feature_importances_,
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=fi_impurity, x="Importance", y="Feature")
plt.title("Random Forest Feature Importance (impurity-based)")
plt.tight_layout()
plt.show()
fi_impurity

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=10, random_state=RANDOM_STATE,
    scoring="neg_mean_absolute_error", n_jobs=-1,
)

fi_perm = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std,
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=fi_perm, x="Importance", y="Feature")
plt.title("Random Forest Feature Importance (permutation)")
plt.tight_layout()
plt.show()
fi_perm

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = cross_validate(
    best_model, X_train, y_train,
    cv=kf,
    scoring={"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error"},
    n_jobs=-1,
)

mae_scores  = -cv_results["test_MAE"]
rmse_scores = -cv_results["test_RMSE"]

print(f"MAE  per fold: {np.round(mae_scores, 4)}")
print(f"MAE  mean={mae_scores.mean():.4f}  std={mae_scores.std():.4f}")
print(f"RMSE per fold: {np.round(rmse_scores, 4)}")
print(f"RMSE mean={rmse_scores.mean():.4f}  std={rmse_scores.std():.4f}")

### Spatial cross-validation (pemeriksaan tambahan)

Random K-fold dapat menempatkan wilayah yang berdekatan pada fold berbeda. Sebagai pemeriksaan yang lebih menantang, kelompokkan koordinat ke grid geografis kasar lalu gunakan `GroupKFold`, sehingga satu grid tidak tersebar ke train dan validation dalam fold yang sama.

Ini bukan satu-satunya skema spatial validation yang mungkin, dan hasilnya bergantung pada ukuran grid. Bandingkan dengan random CV dan jelaskan bahwa keduanya mengukur skenario generalisasi yang berbeda.

In [ ]:
# Grid 0.5-degree is a pragmatic starting point, not a universal geographic boundary.
spatial_groups = (
    (X_train["Latitude"] / 0.5).round().astype(int).astype(str)
    + "_"
    + (X_train["Longitude"] / 0.5).round().astype(int).astype(str)
)

n_spatial_groups = spatial_groups.nunique()
if n_spatial_groups < 5:
    print(f"Spatial GroupKFold skipped: only {n_spatial_groups} groups.")
else:
    spatial_cv = GroupKFold(n_splits=5)
    spatial_results = cross_validate(
        best_model,
        X_train,
        y_train,
        groups=spatial_groups,
        cv=spatial_cv,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2",
        },
        n_jobs=-1,
    )
    spatial_mae = -spatial_results["test_MAE"]
    spatial_rmse = -spatial_results["test_RMSE"]
    spatial_r2 = spatial_results["test_R2"]

    print(f"Spatial groups: {n_spatial_groups}")
    print(f"Spatial CV MAE:  mean={spatial_mae.mean():.4f}, std={spatial_mae.std():.4f}")
    print(f"Spatial CV RMSE: mean={spatial_rmse.mean():.4f}, std={spatial_rmse.std():.4f}")
    print(f"Spatial CV R2:   mean={spatial_r2.mean():.4f}, std={spatial_r2.std():.4f}")

In [ ]:
# Save the exact model selected by GridSearchCV with reproducibility metadata.
artifact = {
    "model": best_model,
    "feature_names": X_train.columns.tolist(),
    "target_name": "MedHouseVal",
    "target_scale_usd": 100_000,
    "model_name": "RandomForestRegressor",
    "best_params": grid_search.best_params_,
    "random_state": RANDOM_STATE,
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "sklearn_version": __import__("sklearn").__version__,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "evaluation": {
        "test_mae": float(tuned_result["MAE Test"]),
        "test_rmse": float(tuned_result["RMSE Test"]),
        "test_r2": float(tuned_result["R2 Test"]),
    },
}

artifact_path = "california_housing_model.joblib"
joblib.dump(artifact, artifact_path)

# Reload and validate the artifact contract.
loaded = joblib.load(artifact_path)
assert loaded["feature_names"] == X_train.columns.tolist()
assert loaded["target_name"] == "MedHouseVal"

sample_input = X_test.loc[:, loaded["feature_names"]].iloc[:5]
sample_predictions = loaded["model"].predict(sample_input)

assert np.isfinite(sample_predictions).all()
print("Artifact saved:", artifact_path)
print("Artifact features:", loaded["feature_names"])
print("Sample predictions:", sample_predictions)

In [ ]:
# Demonstration only: these values are illustrative and are not verified
# to describe an actual California census block group.
input_prediksi = {
    "MedInc": 3.87,
    "HouseAge": 28.0,
    "AveRooms": 5.4,
    "AveBedrms": 1.1,
    "Population": 1425.0,
    "AveOccup": 3.0,
    "Latitude": 35.6,
    "Longitude": -119.5,
}

# Enforce the exact feature order stored with the model.
df_input = pd.DataFrame(
    [[input_prediksi[name] for name in loaded["feature_names"]]],
    columns=loaded["feature_names"],
)

if not np.isfinite(df_input.to_numpy(dtype=float)).all():
    raise ValueError("Input contains NaN or infinite values.")

prediction = float(loaded["model"].predict(df_input)[0])
if not np.isfinite(prediction):
    raise ValueError("Model returned a non-finite prediction.")

prediction_usd = prediction * loaded["target_scale_usd"]

print("Demonstration prediction (not a real property appraisal)")
print(f"MedHouseVal: {prediction:.4f}")
print(f"Approximate USD scale: ${prediction_usd:,.2f}")

### Residual analysis (test set)

Residual analysis membantu melihat apakah error terkonsentrasi pada rentang target tertentu. Ini penting karena dataset memiliki target yang dibatasi pada nilai tinggi.

In [ ]:
# Use the tuned model's test predictions; do not refit on the test set.
y_test_pred = best_model.predict(X_test)
residuals = y_test.to_numpy() - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test_pred, residuals, alpha=0.25)
axes[0].axhline(0, linestyle="--")
axes[0].set_xlabel("Predicted MedHouseVal")
axes[0].set_ylabel("Residual (actual - predicted)")
axes[0].set_title("Residuals vs Predicted")

axes[1].scatter(y_test, y_test_pred, alpha=0.25)
axes[1].plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle="--",
)
axes[1].set_xlabel("Actual MedHouseVal")
axes[1].set_ylabel("Predicted MedHouseVal")
axes[1].set_title("Actual vs Predicted")

plt.tight_layout()
plt.show()

## Conclusion

Angka pada bagian ini sebaiknya diambil dari tabel hasil eksekusi di atas, bukan diketik manual. Setelah seluruh notebook dijalankan ulang, catat:
- Model dengan MAE test terendah.
- MAE, RMSE, dan R² test model terpilih.
- MAE/RMSE cross-validation random.
- Hasil spatial cross-validation sebagai skenario generalisasi geografis tambahan.
- Perbandingan dengan DummyRegressor.

### Interpretasi dan batasan

- `MedHouseVal` adalah median nilai rumah tingkat block group, bukan harga transaksi rumah individual.
- Dataset memiliki batas atas target sekitar 5.0 (≈ US$500.000 pada skala target); error pada segmen bernilai tinggi perlu dianalisis secara khusus.
- Random split dan spatial validation merepresentasikan skenario evaluasi yang berbeda.
- Permutation importance mengukur dampak pengacakan fitur terhadap skor model, bukan hubungan sebab-akibat.
- Input demo adalah contoh sintetis untuk menguji alur inference, bukan bukti akurasi untuk properti nyata.

### Checklist sebelum publikasi

- [ ] Restart kernel dan jalankan semua cell dari awal.
- [ ] Pastikan tidak ada error dan periksa warning.
- [ ] Catat hasil baseline, MAE, RMSE, dan R².
- [ ] Catat hasil random CV dan spatial CV.
- [ ] Pastikan artifact yang dihasilkan adalah artifact yang digunakan backend.
- [ ] Simpan versi dependency dan instruksi menjalankan notebook.
